In [ ]:
#@title Copyright 2019 Google LLC. { display-mode: "form" }
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [2]:
# ============================================================
# Human-disturbance parameters via Google Earth Engine (Colab)
# Directly-available datasets only:
#   - Night-time light intensity   -> NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG
#   - Human population density     -> CIESIN/GPWv411/GPW_Population_Density
#   - Built-up area %               -> ESA/WorldCover/v200
#   - Protected-area %              -> WCMC/WDPA/current/polygons
# (Note: none of these are Landsat products - each comes from its own
#  dedicated dataset, which is why they're the ones directly available.)
# ============================================================

# ---- Cell 1: install/import + auth ----
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project='leafy-clone-504009-p0')   # <-- EDIT: your Earth-Engine-enabled Cloud project

# ---- Cell 2: config ----

COORDS = [   # (lat, lon) - your bird coordinates
    (23.7,      69.4),
    (23.375,    69.714),
    (23.242,    69.6669),
    (23.242,    69.6669),
    (23.166667, 71.75),
    (23.4925,   68.5844),
    (23.03,     69.56),
    (23.355,    69.754),
    (23.242,    69.6669),
]

BUFFERS_M  = [500, 1000]
SCALE      = 100          # coarser than Landsat since VIIRS/GPW are ~500m/1km native
OUTPUT_CSV = 'human_disturbance_parameters.csv'

# ---- Cell 3: build the 4 layers ----

# Night-time light intensity - mean radiance over the most recent full year available
viirs = (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
         .filterDate('2023-01-01', '2023-12-31')
         .select('avg_rad')
         .mean()
         .rename('night_lights_radiance'))

# Human population density - most recent modeled year (2020)
pop_density = (ee.ImageCollection('CIESIN/GPWv411/GPW_Population_Density')
               .filterDate('2020-01-01', '2020-12-31')
               .first()
               .select('population_density')
               .rename('population_density_per_km2'))

# Built-up area % - ESA WorldCover v200, class 50 = "Built-up"
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
built_up_pct = worldcover.eq(50).multiply(100).rename('built_up_pct')

# Protected-area % - rasterize WDPA polygons (1 = inside a protected area, 0 = outside)
wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons')
protected_pct = (ee.Image(0).byte()
                  .paint(wdpa, 1)
                  .multiply(100)
                  .rename('protected_area_pct'))

combined = viirs.addBands(pop_density).addBands(built_up_pct).addBands(protected_pct)

# ---- Cell 4: zonal mean per coordinate, per buffer ----

rows = []
for lat, lon in COORDS:
    pt = ee.Geometry.Point([lon, lat])
    row = {'lat': lat, 'lon': lon}
    for buf in BUFFERS_M:
        stats = combined.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=pt.buffer(buf),
            scale=SCALE,
            maxPixels=1e9
        ).getInfo()
        for k, v in stats.items():
            row[f'{k}_{buf}m'] = v
    rows.append(row)
    print(f'done: {lat}, {lon}')

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
df

# ---- Cell 5 (optional, Colab only): download the CSV ----
# from google.colab import files
# files.download(OUTPUT_CSV)

done: 23.7, 69.4
done: 23.375, 69.714
done: 23.242, 69.6669
done: 23.242, 69.6669
done: 23.166667, 71.75
done: 23.4925, 68.5844
done: 23.03, 69.56
done: 23.355, 69.754
done: 23.242, 69.6669


,lat,lon,built_up_pct_500m,night_lights_radiance_500m,population_density_per_km2_500m,protected_area_pct_500m,built_up_pct_1000m,night_lights_radiance_1000m,population_density_per_km2_1000m,protected_area_pct_1000m
0,23.700000,69.4000,0.000000,0.370610,22.102373,0,0.000000,0.370583,22.102373,0
1,23.375000,69.7140,0.000000,1.181412,22.102373,0,0.295135,0.986725,22.102374,0
2,23.242000,69.6669,86.325301,27.598854,22.102374,0,75.297129,24.236571,22.102374,0
3,23.242000,69.6669,86.325301,27.598854,22.102374,0,75.297129,24.236571,22.102374,0
4,23.166667,71.7500,0.000000,0.760692,121.174683,0,0.000000,0.788497,120.670193,0
5,23.492500,68.5844,1.706832,6.269947,36.343094,0,8.463210,5.633714,36.343095,0
6,23.030000,69.5600,0.000000,0.590569,76.816704,0,3.409948,0.634414,81.540328,0
7,23.355000,69.7540,0.000000,0.535834,22.102377,0,0.000000,0.547042,22.102375,0
8,23.242000,69.6669,86.325301,27.598854,22.102374,0,75.297129,24.236571,22.102374,0


<table class="ee-notebook-buttons" align="left"><td>
<a target="_blank"  href="http://colab.research.google.com/github/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> Run in Google Colab</a>
</td><td>
<a target="_blank"  href="https://github.com/google/earthengine-community/blob/master/guides/linked/ee-api-colab-setup.ipynb"><img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" /> View source on GitHub</a></td></table>

# Earth Engine Python API Colab Setup

This notebook demonstrates how to setup the Earth Engine Python API in Colab and provides several examples of how to print and visualize Earth Engine processed data.

## Import API and get credentials

The Earth Engine API is installed by default in Google Colaboratory so requires only importing and authenticating. These steps must be completed for each new Colab session, if you restart your Colab kernel, or if your Colab virtual machine is recycled due to inactivity.

### Import the API

Run the following cell to import the API into your session.

In [ ]:
import ee

### Authenticate and initialize

Run the `ee.Authenticate` function to authenticate your access to Earth Engine servers and `ee.Initialize` to initialize it. Upon running the following cell you'll be asked to grant Earth Engine access to your Google account. Follow the instructions printed to the cell.

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='my-project')

## Test the API

Test the API by printing the elevation of Mount Everest.

In [ ]:
# Print the elevation of Mount Everest.
dem = ee.Image('USGS/SRTMGL1_003')
xy = ee.Geometry.Point([86.9250, 27.9881])
elev = dem.sample(xy, 30).first().get('elevation').getInfo()
print('Mount Everest elevation (m):', elev)

## Map visualization

`ee.Image` objects can be displayed to notebook output cells. The following two
examples demonstrate displaying a static image and an interactive map.


### Static image

The `IPython.display` module contains the `Image` function, which can display
the results of a URL representing an image generated from a call to the Earth
Engine `getThumbUrl` function. The following cell will display a thumbnail
of the global elevation model.

In [ ]:
# Import the Image function from the IPython.display module.
from IPython.display import Image

# Display a thumbnail of global elevation.
Image(url = dem.updateMask(dem.gt(0))
  .getThumbURL({'min': 0, 'max': 4000, 'dimensions': 512,
                'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']}))

### Interactive map

The [geemap](https://github.com/gee-community/geemap)
library can be used to display `ee.Image` objects on an interactive
[ipyleaflet](https://github.com/jupyter-widgets/ipyleaflet) map.

The following cell provides an example of using the `geemap.Map` object to
display an elevation model.

In [ ]:
# Import the geemap library.
import geemap

# Set visualization parameters.
vis_params = {
  'min': 0,
  'max': 4000,
  'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']}

# Create a map object.
m = geemap.Map(center=[20, 0], zoom=3)

# Add the elevation model to the map object.
m.add_ee_layer(dem.updateMask(dem.gt(0)), vis_params, 'DEM')

# Display the map.
display(m)

## Chart visualization

Some Earth Engine functions produce tabular data that can be plotted by
data visualization packages such as `matplotlib`. The following example
demonstrates the display of tabular data from Earth Engine as a scatter
plot. See [Charting in Colaboratory](https://colab.sandbox.google.com/notebooks/charts.ipynb)
for more information.

In [ ]:
# Import the matplotlib.pyplot module.
import matplotlib.pyplot as plt

# Fetch a Landsat TOA image.
img = ee.Image('LANDSAT/LT05/C02/T1_TOA/LT05_034033_20000913')

# Select Red and NIR bands and sample 500 points.
samp_fc = img.select(['B3','B4']).sample(scale=30, numPixels=500)

# Arrange the sample as a list of lists.
samp_dict = samp_fc.reduceColumns(ee.Reducer.toList().repeat(2), ['B3', 'B4'])
samp_list = ee.List(samp_dict.get('list'))

# Save server-side ee.List as a client-side Python list.
samp_data = samp_list.getInfo()

# Display a scatter plot of Red-NIR sample pairs using matplotlib.
plt.scatter(samp_data[0], samp_data[1], alpha=0.2)
plt.xlabel('Red', fontsize=12)
plt.ylabel('NIR', fontsize=12)
plt.show()